In [1]:
import json
import pandas as pd

In [2]:
with open('big_east.json') as f:
    big_east = json.load(f)

In [3]:
be17 = big_east['2017']


In [4]:
be17

[['Villanova', 5],
 ['Butler', 2],
 ['Creighton', 4],
 ['Marquette', 2],
 ['Seton Hall', 3],
 ['Providence', 2],
 ['Xavier', 3],
 ["St. John's", 2],
 ['Georgetown', 1],
 ['DePaul', 1]]

In [5]:
be17_seed = pd.DataFrame([{"TEAM": x[0], "SEED": be17.index(x) + 1, "POSTSEASON": x[1]} for x in be17])

In [6]:
df17 = pd.read_csv('./data/full/cbb21.csv')

In [7]:
new_columns = ['TEAM',
 'CONF',
 'POSTSEASON',
 'G',
 'W',
 "WIN_PER",
 'ADJOE',
 'ADJDE',
 'BARTHAG',
 'EFG_O',
 'EFG_D',
 'TOR',
 'TORD',
 'ORB',
 'DRB',
 'FTR',
 'FTRD',
 '2P_O',
 '2P_D',
 '3P_O',
 '3P_D',
 'ADJ_T',
 'WAB',
 'SEED',
 "POWER",
 'YEAR']

In [8]:
df_be17 = df17.drop(columns = ['SEED']).merge(be17_seed, on='TEAM', how='inner').sort_values('SEED').reset_index(drop=True)


In [9]:
df_be17['WIN_PER'] = df_be17['W']/df_be17['G']
df_be17['POWER'] = 1
df_be17['YEAR'] = 2017

In [10]:
df_be17 = df_be17[new_columns]

In [11]:
def round_assign(x):
    if x['POSTSEASON'] == 5 : return [1,1,1,1,1]
    elif x['POSTSEASON'] == 4 : return [1,1,1,1,0]
    elif x['POSTSEASON'] == 3 : return [1,1,1,0,0]
    elif x['POSTSEASON'] == 2 : return [1,1,0,0,0]
    elif x['POSTSEASON'] == 1 : return [1,0,0,0,0]
    else: return [0,0,0,0,0]

In [12]:
def assign_dummy(df):
    dummy_df = []
    for x in range(0,len(df)):
        dummy_df.append(round_assign(df.iloc[x]))
    dummy_df = pd.DataFrame(dummy_df)
    dummy_df = dummy_df.rename(columns={0: "R1", 1: "R2", 2: "R3",3: "R4", 4:"R5"})
    return dummy_df

In [13]:
df_be17 =df_be17.join(assign_dummy(df_be17))

In [14]:
df_be17

,TEAM,CONF,POSTSEASON,G,W,WIN_PER,ADJOE,ADJDE,BARTHAG,EFG_O,...,ADJ_T,WAB,SEED,POWER,YEAR,R1,R2,R3,R4,R5
0,Villanova,BE,5,22,16,0.727273,117.9,96.7,0.9076,52.8,...,65.1,3.2,1,1,2017,1,1,1,1,1
1,Butler,BE,2,25,10,0.400000,102.3,97.9,0.6237,47.5,...,64.6,-3.6,2,1,2017,1,1,0,0,0
2,Creighton,BE,4,28,20,0.714286,114.4,94.3,0.9025,55.7,...,69.1,3.5,3,1,2017,1,1,1,1,0
3,Marquette,BE,2,27,13,0.481481,107.4,95.8,0.7885,50.6,...,67.9,-2.1,4,1,2017,1,1,0,0,0
4,Seton Hall,BE,3,27,14,0.518519,109.5,94.8,0.8401,50.0,...,67.1,-1.0,5,1,2017,1,1,1,0,0
5,Providence,BE,2,26,13,0.500000,107.7,94.7,0.8151,48.3,...,66.7,-1.8,6,1,2017,1,1,0,0,0
6,Xavier,BE,3,21,13,0.619048,109.8,96.8,0.8087,52.1,...,68.4,-0.6,7,1,2017,1,1,1,0,0
7,St. John's,BE,2,27,16,0.592593,110.7,98.3,0.7974,50.4,...,73.8,-0.6,8,1,2017,1,1,0,0,0
8,Georgetown,BE,1,25,13,0.520000,108.5,93.2,0.8516,49.6,...,69.5,-1.0,9,1,2017,1,0,0,0,0
9,DePaul,BE,1,19,5,0.263158,98.9,94.3,0.6346,46.1,...,70.2,-5.8,10,1,2017,1,0,0,0,0


In [15]:
train_columns = [
 'G',
 'W',
 "WIN_PER",
 'ADJOE',
 'ADJDE',
 'BARTHAG',
 'EFG_O',
 'EFG_D',
 'TOR',
 'TORD',
 'ORB',
 'DRB',
 'FTR',
 'FTRD',
 '2P_O',
 '2P_D',
 '3P_O',
 '3P_D',
 'ADJ_T',
 'WAB',
 'SEED'
 ]

In [16]:
def get_upset_differences(a,b):
    """
    Takes two rows from one of the region databases and finds the difference between the two columns 

    a: higher seed
    b: lower seed'

    output: list of db difference rows
    """
    
    listA = []
    for x in range(3,24):
        diff = a.iloc[x] - b.iloc[x]
        listA.append(diff)
    return listA

In [17]:
def get_target_variable(df, nxt_round_num, rnd_name):
    """
    This function used to get the target variable attributes from the correct column
    The list is reversed because we want to see if the lower seed wins

    df: dataframe used
    nxt_round_num: number of teams in the next round
    rnd_name: the column name of the dummy round variable of the next round

    output: list of target variable attributes (will be its own column)
    """
    
    listB = list(df[rnd_name])
    listB.reverse()
    listB = listB[0:nxt_round_num]
    return listB

In [18]:
def create_training_record(df, matchup_num, reseed = True):
    """
    This function takes the dataframe, and creates a new dataset of differences that can be trained
    
    df: dataframe of the round being processed
    matchup_num: the number of matchups
    """
    
    listDF = []
    for y in range(0,matchup_num):
        test_upsetH = df.iloc[y]
        test_upsetL = df.iloc[-(y+1)]
        listDF.append(get_upset_differences(test_upsetH,test_upsetL))
    if reseed:
        return pd.DataFrame(listDF, columns = train_columns).sort_values(by = ["SEED"])
    else:
        return pd.DataFrame(listDF, columns = train_columns).sort_values(by = ["SEED"])

In [19]:
def creation(df, next_rounds):
    """
    This function takes the current round's dataframe and the next round string name to use the previous functions to build the training df

    df: current round's dataframe
    next_rounds
    """
    
    y = pd.DataFrame(columns = train_columns)
    asd = []
    nxt_round_num = int(len(df)/2)
    upset= get_target_variable(df,nxt_round_num, next_rounds)
    for x in range(0,nxt_round_num):
        h = df.iloc[x]
        l = df.iloc[(-x-1)]
        if upset[x] == 1:
            asd.append(l)
        if upset[x] == 0:
            asd.append(h)
    next_df = pd.DataFrame(asd)
    y = pd.concat([y, create_training_record(df,nxt_round_num)], ignore_index=True)
    y["TRAIN"] = upset
    y.TRAIN = y.TRAIN.astype(int)
    return y, next_df

In [20]:
def create_train(a, next_round):
    df = pd.DataFrame(columns = train_columns)
    df_next = []
    for x in a:
        train, next_df = creation(x,next_round)
        df = pd.concat([df,train], ignore_index = True)
        df_next.append(next_df)
    return df, df_next


In [21]:
df = pd.DataFrame(columns = train_columns)
df_next = []

In [22]:
round_name = ["R2", "R3", "R4", "R5"]

In [23]:
def bye_split(df):
    team_cnt = len(df)
    x = 0
    while 2**x < team_cnt:
        binary_rnd = 2**x 
        x += 1
    
    second_round = binary_rnd + (binary_rnd - team_cnt)
    
    return df.iloc[:second_round],df.iloc[second_round:]

In [24]:
df, r1_df= bye_split(df_be17)

In [25]:
r1_df

,TEAM,CONF,POSTSEASON,G,W,WIN_PER,ADJOE,ADJDE,BARTHAG,EFG_O,...,ADJ_T,WAB,SEED,POWER,YEAR,R1,R2,R3,R4,R5
6,Xavier,BE,3,21,13,0.619048,109.8,96.8,0.8087,52.1,...,68.4,-0.6,7,1,2017,1,1,1,0,0
7,St. John's,BE,2,27,16,0.592593,110.7,98.3,0.7974,50.4,...,73.8,-0.6,8,1,2017,1,1,0,0,0
8,Georgetown,BE,1,25,13,0.520000,108.5,93.2,0.8516,49.6,...,69.5,-1.0,9,1,2017,1,0,0,0,0
9,DePaul,BE,1,19,5,0.263158,98.9,94.3,0.6346,46.1,...,70.2,-5.8,10,1,2017,1,0,0,0,0


In [84]:
df_past_R1 = []
df_past_R1.append(df_be17)

In [86]:
train_master = pd.DataFrame(columns = train_columns)

for x in range(len(round_name)):
    print(x)
    if x == 0:
        if len(bye_split(df_be17)[1]) == 0:
            print('e')
            pass
        else:
            print("here")
            train_df, first_round = bye_split(df_be17)
            holder = create_train([first_round], round_name[x])
            df_past_R2 = pd.concat([train_df, holder[1][0]], ignore_index=True)
            train_master = pd.concat([train_master,holder[0]], ignore_index = True)

            continue
    
    holder = create_train([globals()["df_past_R" + '%s' % (x+1)]], round_name[x])
    globals()["df_past_R" + '%s' % (x+2)] = holder[1]
    train_master = pd.concat([train_master, holder[0]], ignore_index=True)

0
here
1
2


/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_74527/2317163237.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  y = pd.concat([y, create_training_record(df,nxt_round_num)], ignore_index=True)
/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_74527/949454954.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,train], ignore_index = True)
/var/folders/sp/zh4t9j256z73nrrj_cn8r_yc0000gp/T/ipykernel_74527/4259792558.py:14: FutureWarning: The behavior of Da

TypeError: list indices must be integers or slices, not str

In [ ]:
get_target_variable(df_past_R3, 4, "R4")

[0, 1, 0, 1]

In [66]:
type(holder[1])

list